# Original actor-feature FiLM: EE oscillation audit
Offline analysis of each checkpoint's own Exp2 recordings, primarily pushing/pulling. Selection requires recorded `film_v02_enabled=False`; V02 and non-FiLM recordings are excluded. No policy input perturbations, simulation, training, or model construction.
This tests **consistency with amplification**, not causality. Gait, estimator errors, trunk activity, base motion and actuator saturation remain alternative explanations.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'notebooks').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'notebooks'))
import exp3_film_oscillation as analysis
CONFIG = {
    # Figure typography in points; requires installed Times New Roman.
    "figure_title_font_size": 12,
    "title_font_size": 11,
    "axis_label_font_size": 10,
    "tick_font_size": 9,
    "legend_font_size": 9,
    "annotation_font_size": 8,
    'recording_roots': [str(ROOT / 'evaluation/exp2')],
    'physical_comparison': True, 'comparison_recording_roots': None,  # None reuses recording_roots.
    'comparison_tasks': ['b1z1_pact', 'b1z1_unifp_original', 'b1z1_ppo_pos'],
    'comparison_checkpoints': {},  # Optional {task: [recorded SHA256, ...]}; otherwise discover each method.
    'comparison_window_seconds': 2.0, 'comparison_nperseg': 32,  # Identical bins/bands for every method.
    'comparison_plot_windows': 12, 'comparison_max_windows': None,  # Budget for both severity groups across three methods/two tasks.
    'plot_action_references': True,  # Actual / fixed / unmodulated physical-head spectra.
    'tasks': ['b1z1_pact'],  # None discovers every compatible original-FiLM task.
    'checkpoints': None,  # Optional list of recorded SHA256 hashes.
    'checkpoint_paths': {},  # Optional {recorded_sha256: local_checkpoint_path}.
    'evaluation_seeds': [0, 1, 2],
    'profiles': ['pushing', 'pulling'],
    'loading': ['loaded', 'unloaded'],
    'trial_ids': None,  # e.g. [0, 1] for a small audit.
    'sustained_window_seconds': 4.0,
    'event_window_seconds': [-0.5, 2.0],
    'transient_margin_seconds': 0.25,
    'minimum_window_seconds': 1.25,  # Includes short loaded plateaus; not a low-frequency guarantee.
    'reference_stationarity_tolerance': 0.02,  # Max change in each native sphere coordinate.
    'nperseg': 64, 'overlap_fraction': 0.5, 'min_spectral_segments': 4,
    'segment_length_policy': 'adaptive', 'min_nperseg': 24,  # Or fixed; shorten only to retain >=4 averages.
    'minimum_cycles_per_window': 3,  # Frequencies below this or Welch resolution are unavailable.
    'moving_reference_mode': 'linear_detrended',  # Separate descriptive residual links; 'exclude' suppresses them.
    'temporal_ee_axes': [0, 1, 2],  # All Cartesian axes crossed with all spherical components, not one-to-one.
    'temporal_joint_names': {},  # Optional named lists; default all arm / front-right leg joints. Grouped spectra use all legs.
    'joint_groups': {},  # Optional exact recorded name lists; defaults z1_* (not gripper), FR_/FL_/RR_/RL_.
    'load_direction_min_n': 1., 'load_direction_tolerance_deg': 10.,
    'frequency_band_hz': [0.5, 12.0],  # None uses all positive Welch frequencies.
    'coherence_threshold': 0.6, 'max_lag_seconds': 0.5,
    'feature_epsilon': 1e-8,
    'head_parity_atol': 3e-5, 'head_parity_rtol': 3e-4,  # Strict CPU/GPU raw-action check; failures are reported, not hidden.
    'precision_aware_attribution': True,  # Accept explicitly bounded, validated reduced-precision arithmetic.
    'allow_unverified_attribution': True,  # Permissive: unexplained algebraic results remain visibly UNVERIFIED.
    'minimum_comparison_windows': 6, 'error_match_bin_m': 0.02,
    'pca': True, 'pca_samples': 1000, 'sampling_seed': 0,
    'pca_color': 'signed_error',  # Or load_stage / oscillation_phase (qualitative Hilbert phase).
    'tsne': False, 'tsne_perplexity': 30,  # Optional sklearn; qualitative only.
    'max_plot_windows': 12, 'max_windows': None,  # max_windows limits smoke runs only.
    'max_plot_relationships': 8,  # Figure-only cap, attributed links first; CSV keeps every link and named component.
    'output_dir': str(ROOT / 'analysis/exp3_film_oscillation'),
}
CONFIG.update(json.loads(os.environ.get('FILM_OSCILLATION_CONFIG', '{}')))


## Timing, coordinates, and spectral interpretation
Cartesian error is measured minus reference in the yaw-aligned task frame, in metres. Conditioning is command minus estimate: normalized body `[vx, vy, yaw_rate]` and spherical `[radius, elevation, azimuth]`. Measured conditioning is reconstructed at actor time **t**, using the recorded spherical/world reference pair to recover the workspace origin. Successor errors/motion remain at **t+1**.
Only contiguous valid pre-failure segments are analyzed, within each recorded load/motion stage. Short/failed windows are reported, not padded. Adaptive Hann Welch shortens segments to retain at least four overlapping averages (not four independent samples). Each window exports duration, segment count, resolution and usable frequency range; requested low frequencies below these limits are not interpreted. Stationary windows use constant detrending. Moving-reference tracking residuals and linked channels use linear detrending in a **separate nonstationary diagnostic**, never pooled with stationary results. Event windows retain descriptive spectra but no coherence. Lag peaks are exploratory: positive lag means the right signal follows the left; successor motion includes its +dt timestamp offset.
Temporal links cross all Cartesian error axes with native spherical components and full head-projected changes at named joints; no arbitrary feature coordinate or one-to-one Cartesian/spherical mapping. Contacts/base motion provide competing periodic explanations, compared descriptively within stages, not a causal adjustment. Load projections use the exported world basis. Arm/leg groups exclude grippers.
`gamma` is residual scaling: `post = (1+gamma)*pre + beta`. Temporal change is exactly `(1+gamma_previous)*delta_pre + delta_gamma*pre_current + delta_beta`. The last two terms form the parameter-change contribution. Their spectral powers are **not additive**, because cross terms remain.

In [ ]:
result = analysis.run_analysis(CONFIG)
print(f"Analyzed {len(result['windows'])} windows from {len(result['provenance'])} recordings")
summary = pd.DataFrame(result['summary'])
if len(summary):
    metrics = ['ee_cartesian_error/oscillatory_rms', 'relative_modulation_rms', 'post_pre_oscillation_ratio', 'parameter_feature_change_ratio', 'position_actual_arm/oscillatory_rms', 'position_fixed_arm/oscillatory_rms', 'position_unmodulated_arm/oscillatory_rms', 'torque_actual_arm/oscillatory_rms', 'torque_fixed_arm/oscillatory_rms', 'torque_unmodulated_arm/oscillatory_rms']
    display(summary[summary.metric.isin(metrics)])
display(pd.DataFrame(result['exclusions']))


## Recorded feature/action associations
Figures show deterministically selected windows, covering task, load/stage, sustained/event type, checkpoint/seed and lower/higher measured oscillatory RMS within comparable conditions. Selection prefers different trials; the budget may leave categories omitted, explicitly exported. It is not a representative random population sample. Tables average windows within trials, then trials within evaluation seeds, then evaluation seeds within each checkpoint; overlapping windows are not independent replicates. Stationary and moving-reference modes remain separate. Sample SD is undefined for one seed. Checkpoints stay separate; evaluation seeds are not independent training runs.
Head checks distinguish strict parity, **precision-explained approximate** attribution, and **unverified algebraic-only** results. Saved data are float32; a CPU 10-fraction-bit operand model (nearest, ties away) explains observed head discrepancies without raising tolerances. Acceptance requires matching actions, substantial residual reduction and an operand/accumulation error bound. GPU precision metadata were not recorded: this supports an arithmetic model, not proof of its backend setting. Per-joint/channel errors and scales are in action_attribution_audit.csv. Permissive mode retains unexplained results flagged; do not treat them as validated decomposition. Wrong checkpoint hashes remain fatal.
Actual, window-fixed `(1+mean(gamma))*pre+mean(beta)`, and unmodulated head references are algebraic only. Persistent/varying effects include spectral cross-terms; powers are not additive. Position outputs are scaled offsets [rad]; FF [Nm] precedes motor/branch weights. Nominal requests and clipped execution stay separate. No alternate trajectory was executed.
High/low comparisons are stratified by checkpoint/profile/load/motion stage, analysis mode and attribution quality, with explicit error-magnitude bins. `trial_sd` describes variation among trial averages; `sd` describes variation among evaluation-seed averages (undefined for one seed). Overlapping windows are averaged within trials, not treated as independent samples. Paired fixed-minus-unmodulated and actual-minus-fixed RMS differences describe amplitude changes, not additive component energies. Phase at an observed EE peak is unavailable below the coherence threshold. PCA uses one centered pooled pre/post basis per checkpoint with paired points; explained variance is exported. t-SNE is disabled by default and cannot establish feedback amplification.

In [ ]:
OUT = Path(CONFIG['output_dir'])
display(pd.DataFrame(result['coverage']))
display(pd.DataFrame(result['plot_omitted_categories']))
for selected in result['plot_selection']:
    print({k: selected[k] for k in ['profile', 'loading', 'stage', 'checkpoint', 'evaluation_seed', 'trial_id', 'actual_start', 'actual_end', 'severity_group']})
    for suffix in ('traces', 'spectra'):
        display(Image(filename=str(OUT / f"window-{selected['window_id']:05d}-{suffix}.png")))
    references = OUT / f"window-{selected['window_id']:05d}-action-references.png"
    if references.exists(): display(Image(filename=str(references)))
display(pd.DataFrame(result['pca']).T)
for path in sorted(OUT.glob('pca-*.png')):
    display(Image(filename=str(path)))
print('Exports:', OUT)


## Matched physical oscillations and competing explanations
Physical comparisons discover PILOT/UniFP/Pos independently of FiLM fields. Identical recorded protocols, seeds and scheduled windows are matched; different PD gains remain in provenance. Fixed-duration windows use linear detrending and the same Welch bins/band across methods. Bias, drift and residual oscillatory RMS/peak-to-peak remain separate. Missing/incomplete windows are excluded and counts/failures reported, not padded. Without unloaded controls, pre-load versus loaded is confounded by trajectory and time.
Velocity decomposition uses simultaneous actor-time world quantities: `v_EE = v_base + omega_base x (p_EE-p_base) + v_relative`. The current Isaac Lab recorder supplies root-link position/velocity and gripper-link velocity (not CoM velocities); historical recordings must share that contract. The relative term includes arm motion and any unmodelled compliance/reference-point error; it is not independently validated J*qdot. Negative base/relative cross-power is consistent with attenuation. Contacts, PD/FF channel use, gains and shared spectral peaks are associations, not established causes. Statistics average windows within trial then trials within evaluation seed, keeping checkpoints separate. High/low oscillation differs by construction; explanatory signals and seed-level uncertainty matter.


In [ ]:
physical = result.get('physical_comparison', {})
comparison = pd.DataFrame(physical.get('matched', []))
if len(comparison):
    physical_table = comparison[(comparison.signal == 'ee_error') & (comparison.scheduled_load_n > 0)].copy()
    from b1z1_paper_names import DISPLAY_NAMES
    physical_table['method'] = physical_table.task.map(lambda name: DISPLAY_NAMES.get(name, name))
    display(physical_table.pivot(index=['method','checkpoint','profile','loading','stage','axis','protocol_hash','spectral_contract'], columns='metric', values=['mean','std']))  # Bias [m], drift [m/s], RMS/peak [m], power [m²], frequency [Hz].
    display(physical_table[['method','profile','stage','axis','metric','count','trial_seed_pairs','windows']])
else:
    print('Matched comparative conclusions unavailable; inspect method-specific physical_summary.csv and diagnostics.')
display(pd.DataFrame(physical.get('provenance', [])))  # Includes failures, attempted trials, actual gains and checkpoint identities.
print('\n'.join(physical.get('diagnostics', [])))
audit_path = Path(CONFIG['output_dir']) / 'action_attribution_audit.csv'
if audit_path.exists() and audit_path.stat().st_size > 1:
    audit = pd.read_csv(audit_path)
    display(audit.groupby(['task', 'head', 'status'])[['max_abs_error', 'precision_residual_max', 'error_bound_max']].max())
display(pd.DataFrame(result['evidence']))
if len(summary):
    reference_rows = summary[summary.metric.str.contains('actual_|fixed_|unmodulated_|persistent_varying_cross_power|minus_')].copy()
    reference_rows.to_csv(OUT / 'action_reference_summary.csv', index=False)
    display(reference_rows[reference_rows.metric.str.endswith(('oscillatory_rms','minus_unmodulated_rms','minus_fixed_rms','cross_power'))])
high_low = OUT / 'high_low_windows.csv'
if high_low.exists() and high_low.stat().st_size > 1: display(pd.read_csv(high_low))
print(result['conclusion'])
for selected in physical.get('plot_selection', []):
    display(Image(filename=str(Path(CONFIG['output_dir']) / 'physical_comparison' / f"physical-{selected['window_id']:05d}.png")))


## Dependencies and outputs
CPU: NumPy, SciPy, pandas, matplotlib, Jupyter. Optional PyTorch only deserializes frozen head weights; optional sklearn supplies t-SNE. Isaac Lab and BARD are never initialized.
`summary.json` stores configuration, provenance, units, exclusions, window completeness and checkpoint/seed summaries. `coverage.csv` lists each planned stage and analysis availability; per-window spectral plans remain in JSON. `plot_selection.csv` and `plot_omitted_categories.json` audit selected coverage. CSVs retain per-axis/per-joint statistics, high/low comparisons and frequency/lag links. `window-*.npz` contains timestamped signals and component PSDs, with PDF/PNG figures. A second bounded pass rereads only selected windows for plotting. `COMPLETE` is written last. Use a fresh output directory; no raw recording is modified.